# Toolboxes CRUD Basics `@azure/ai-projects`

This notebook demonstrates how to perform CRUD operations on **Toolboxes** using the `AIProjectClient`.

Toolboxes are currently a preview feature. In the JS SDK, you access these operations via `project.toolboxes`.

It mirrors the [`toolboxesCrud.ts`](./toolboxesCrud.ts) sample and runs the **locally built** `@azure/ai-projects` from this repo.

## Prerequisites

1. **Build the package first** so `dist/` is current: `cd sdk/ai/ai-projects && pnpm build`
2. **tslab kernel** installed and registered (`npm install -g tslab` then `tslab install`); select the **TypeScript** (tslab) kernel.
3. **Launch VS Code / Jupyter from `sdk/ai/ai-projects/`** so Node resolves the local `@azure/ai-projects`.
4. **`az login`** completed so `DefaultAzureCredential` can authenticate.
5. **Environment variables**: `FOUNDRY_PROJECT_ENDPOINT`.

Run the cells in order (top to bottom); state is shared across cells.

In [1]:
// Imports and configuration
import type { MCPToolboxTool, ToolboxToolUnion } from "@azure/ai-projects";
import { RestError } from "@azure/ai-projects";
import { AIProjectClient } from "@azure/ai-projects";
import { DefaultAzureCredential } from "@azure/identity";

const projectEndpoint = process.env["FOUNDRY_PROJECT_ENDPOINT"] ?? "<project endpoint>";
const toolboxName = "mcp";

In [2]:
// Create the AI Project client
const project = new AIProjectClient(projectEndpoint, new DefaultAzureCredential());

In [ ]:
// Clean up any existing toolbox with this name
try {
  await project.toolboxes.delete(toolboxName);
  console.log(`Toolbox \`${toolboxName}\` deleted`);
} catch (e) {
  if (!(e instanceof RestError && e.statusCode === 404)) {
    throw e;
  }
  console.log(`No existing toolbox \`${toolboxName}\` to delete`);
}

In [3]:
// Define tools for the toolbox and create a new toolbox version
const tools: ToolboxToolUnion[] = [
  {
    type: "mcp",
    server_label: "api_specs",
    server_url: "https://gitmcp.io/Azure/azure-rest-api-specs",
    require_approval: "never",
  } satisfies MCPToolboxTool,
];

const created: any = await project.toolboxes.createVersion(toolboxName, tools, {
  description: "Example toolbox created by the @azure/ai-projects sample.",
  metadata: { status: "created" },
});
const status = created.metadata?.["status"] ?? "unknown status";
console.log(`Toolbox: ${created.name} (tools: ${created.tools.length}) (status: ${status})`);

Toolbox: mcp (tools: 1) (status: created)


In [4]:
// Retrieve the toolbox
const fetched: any = await project.toolboxes.get(toolboxName);
console.log(`Retrieved toolbox: ${fetched.name} (${fetched.id})`);

Retrieved toolbox: mcp (toolbox_199a2adb5b77e520c7a294d117e5c45679a07b3f)


In [5]:
// List toolboxes
const toolboxes = [];
const listToolboxes = async () => {
  for await (const item of project.toolboxes.list({ limit: 10 })) {
    toolboxes.push(item);
  }
};
await listToolboxes();
console.log(`Found ${toolboxes.length} toolboxes`);
for (const item of toolboxes) {
  console.log(`  - ${item.name} (${item.id})`);
}

Found 11 toolboxes
  - mcp (toolbox_199a2adb5b77e520c7a294d117e5c45679a07b3f)
  - toolbox_with_skill (toolbox_2b7611f09ac781bc66ca29bcb9380debf124754e)
  - toolbox_with_mcp_tool (toolbox_3d00c324d4218fa8d5d3f0e6899f3a7be09916d8)
  - toolbox_with_skill-6 (toolbox_e4ea0f98ac640e8d8ebf847d3922e1f93326a28b)
  - toolbox_with_skill-5 (toolbox_aa4e4c203d72cb6facb8fc4f34dff3f445bbf805)
  - toolbox_with_skill-4 (toolbox_6b6db4b823675626399a1d5f285fc02e2a756460)
  - toolbox_with_skill-3 (toolbox_9cc7d4dcbde10d8044a1b552c560f5be29c69261)
  - toolbox-user-isolation-8db00c2a (toolbox_91c81c13a40574176b31467b9d05b29c48cdad75)
  - toolbox-user-isolation-9f8d30c4 (toolbox_decf630ce63555da0ff0c76bcc5f880f25506337)
  - sample-toolbox-with-skill (toolbox_a9c47739e9cf20cdd9d28c9fa99115db52e1a205)
  - t (toolbox_80d1d5d30af91503694b24480f3af477ee2fc401)
  - mcp (toolbox_199a2adb5b77e520c7a294d117e5c45679a07b3f)
  - toolbox_with_skill (toolbox_2b7611f09ac781bc66ca29bcb9380debf124754e)
  - toolbox_with_mcp_t

In [6]:
// Delete the toolbox
await project.toolboxes.delete(toolboxName);
console.log("Toolbox deleted");

Toolbox deleted
